In [18]:
# This allows us to import from all folders one level up from notebooks folder - run 1 time
import sys
from pathlib import Path

print('All paths pre-append')
for i,p in enumerate(sys.path):
    print(f"{i}: {p}")
print('-'*100)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print('All paths post-append')
for i,p in enumerate(sys.path):
    print(f"{i}: {p}")


All paths pre-append
0: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python311.zip
1: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11
2: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/lib-dynload
3: 
4: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/site-packages
----------------------------------------------------------------------------------------------------
All paths post-append
0: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python311.zip
1: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11
2: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/lib-dynload
3: 
4: /Users/irabandutta/Developer/2026-08-llm-from-scratch/venv/lib/python3.11/site-packages
5: /Users/irabandutta/Developer/2026-08-llm-from-scratch


# Imports

In [20]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Tuple
from src.model.llm_config import LLMConfig
from src.model.llm import LLM

# DataLoader

In [ ]:
class TokenDataLoader:
    def __init__(self, B:int, T:int, binary_file_path:str, dtype:np.dtype, debug:bool=False):
        self.B = max(1, B)
        self.T = max(4, T)
        if not debug:
            self.tokens = np.memmap(
                filename=binary_file_path,
                dtype=dtype,
                mode='r'
            )
        else:
            self.tokens = np.arange(1, 101)
        if len(self.tokens)<=(self.B)*(self.T):
            raise ValueError(
                f"Current values of batch size {B} and seq length {T} are too large for dataset, please reduce either or both"
            )
        self.curr_idx = 0

    def next_batch(self) -> Tuple[torch.tensor]:
        B, T = (self.B), (self.T)

        buffer = self.tokens[self.curr_idx:self.curr_idx+(B*T+1)]
        x = torch.tensor(buffer[:-1]).view(B, T)
        y = torch.tensor(buffer[1:]).view(B, T)

        # Covert x,y from uint16 to int32 and int64
        x = x.int()
        y = y.long()
        
        self.curr_idx += B*T
        if self.curr_idx+(B*T+1) > len(self.tokens):
            self.curr_idx=0

        return x, y


binary_file_path = '../data/tinystories/processed/train.bin'
B = 4
T = 8
tok_dl = TokenDataLoader(B, T, binary_file_path, np.uint16, False)
print('#Tokens (in M):', len(tok_dl.tokens)/10e6)
# print('-'*100)
# steps = 2
# for _ in range(steps):
#     x, y = tok_dl.next_batch()
#     print('x:\n', x)
#     print('y:\n', y)    
# print('-'*100)

#Tokens (in M): 47.1872517


In [37]:
# Get a sample batch of tokens of shape (B, T) and overfit model on that batch

x, y = tok_dl.next_batch()
print(x.shape, y.shape)

print(x.dtype, y.dtype)

torch.Size([4, 8]) torch.Size([4, 8])
torch.uint16 torch.uint16


# Instantiate model

In [22]:
# ======== DEFINE Model ========
ctx_len = T
d_model = 64

llm_config = LLMConfig(
    vocab_size=50257,
    ctx_len=ctx_len,
    d_model=d_model, 
    n_layer=2,
    ff_ratio=4,
    dropout=0.0,
    eps=1e-5,
    position_embedding='sinusoidal',
    rotary_embedding=False,
    attention='mha',
    normalization='layernorm',
    n_heads=4, 
    n_groups=None,
    use_flash=False, 
    attn_debug=False
)
print(llm_config)
print('-'*50)

# Detect and resolve device
device = 'cpu'
if torch.cuda.is_available():
    device='cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device='mps'

print("Device found:", device)
print('-'*50)


# Instantiate model
model = LLM(llm_config)
model = model.to(device)
print(f"Model instantiated and moved to {device}")
print('-'*50)

LLMConfig(vocab_size=50257, ctx_len=8, d_model=64, n_layer=2, ff_ratio=4, dropout=0.0, eps=1e-05, position_embedding='sinusoidal', rotary_embedding=False, attention='mha', normalization='layernorm', n_heads=4, n_groups=None, use_flash=False, attn_debug=False)
--------------------------------------------------
Device found: mps
--------------------------------------------------
Model instantiated and moved to mps
--------------------------------------------------


In [32]:
# Move x and y to device
print(x.device, y.device)
x, y = x.to(device), y.long().to(device)
print(x.device, y.device)
print(x.dtype, y.dtype)

mps:0 mps:0
mps:0 mps:0
torch.uint16 torch.int64


In [36]:
x.int()

tensor([[ 4259,   534, 10147,   526,   198,   198, 41631,    11],
        [  484,  4888,   262, 17598,   290,   384, 19103,   262],
        [ 4936,   319, 20037,   338, 10147,    13,   632,   373],
        [  407,  2408,   329,   606,   780,   484,   547,  7373]],
       dtype=torch.int32)

In [33]:
# 1 forward pass - track init loss
logits, loss = model(x, y)

RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got MPSUInt16Type instead (while checking arguments for embedding)